In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count

spark = SparkSession.builder.appName("DataQualityChecks").getOrCreate()

In [0]:
customers = spark.table("retail_silver.customers")

orders = spark.table("retail_silver.orders")

products = spark.table("retail_silver.products")

order_items = spark.table("retail_silver.order_items")

In [0]:
customers.filter(
    col('customer_id').isNull()
).show()

+-----------+--------------+--------------+--------------+-----------------+---------------+-------------+--------------+----------------+
|customer_id|customer_fname|customer_lname|customer_email|customer_password|customer_street|customer_city|customer_state|customer_zipcode|
+-----------+--------------+--------------+--------------+-----------------+---------------+-------------+--------------+----------------+
+-----------+--------------+--------------+--------------+-----------------+---------------+-------------+--------------+----------------+



In [0]:
customers.groupBy("customer_id") \
.count() \
.filter(col("count") > 1) \
.show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [0]:
customers.filter(
    col('customer_email').isNull()
).show()

+-----------+--------------+--------------+--------------+-----------------+---------------+-------------+--------------+----------------+
|customer_id|customer_fname|customer_lname|customer_email|customer_password|customer_street|customer_city|customer_state|customer_zipcode|
+-----------+--------------+--------------+--------------+-----------------+---------------+-------------+--------------+----------------+
+-----------+--------------+--------------+--------------+-----------------+---------------+-------------+--------------+----------------+



In [0]:
order_items.filter(
    col("order_item_subtotal") < 0
).show()

+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+
|order_item_id|order_item_order_id|order_item_product_id|order_item_quantity|order_item_subtotal|order_item_product_price|
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+



In [0]:
products.filter(
    col("product_price")<= 0
).show()

+----------+-------------------+--------------------+-------------------+-------------+--------------------+
|product_id|product_cateogry_id|        product_name|product_description|product_price|       product_image|
+----------+-------------------+--------------------+-------------------+-------------+--------------------+
|        38|                  3|Nike Men's Hyperv...|               NULL|          0.0|http://images.acm...|
|       388|                 18|Nike Men's Hyperv...|               NULL|          0.0|http://images.acm...|
|       414|                 19|Nike Men's Hyperv...|               NULL|          0.0|http://images.acm...|
|       517|                 24|Nike Men's Hyperv...|               NULL|          0.0|http://images.acm...|
|       547|                 25|Nike Men's Hyperv...|               NULL|          0.0|http://images.acm...|
|       934|                 42|Callaway X Hot Dr...|               NULL|          0.0|http://images.acm...|
|      1284|       

In [0]:
orders.select('order_status').distinct().show()

+---------------+
|   order_status|
+---------------+
|         CLOSED|
|PENDING_PAYMENT|
|       COMPLETE|
|     PROCESSING|
| PAYMENT_REVIEW|
|        PENDING|
|        ON_HOLD|
|       CANCELED|
|SUSPECTED_FRAUD|
+---------------+



In [0]:
valid_status = [
    "COMPLETE",
    "PENDING",
    "PROCESSING",
    "CLOSED",
    "SUSPECTED_FRAUD",
    "ON_HOLD",
    "PAYMENT_REVIEW",
    "CANCELED"
]

orders.filter(
    ~col("order_status").isin(valid_status)
).show()

+--------+-------------------+-----------------+---------------+
|order_id|         order_date|order_customer_id|   order_status|
+--------+-------------------+-----------------+---------------+
|       2|2013-07-25 00:00:00|              256|PENDING_PAYMENT|
|       9|2013-07-25 00:00:00|             5657|PENDING_PAYMENT|
|      10|2013-07-25 00:00:00|             5648|PENDING_PAYMENT|
|      13|2013-07-25 00:00:00|             9149|PENDING_PAYMENT|
|      16|2013-07-25 00:00:00|             7276|PENDING_PAYMENT|
|      19|2013-07-25 00:00:00|             9488|PENDING_PAYMENT|
|      23|2013-07-25 00:00:00|             4367|PENDING_PAYMENT|
|      27|2013-07-25 00:00:00|             3241|PENDING_PAYMENT|
|      30|2013-07-25 00:00:00|            10039|PENDING_PAYMENT|
|      33|2013-07-25 00:00:00|             5793|PENDING_PAYMENT|
|      40|2013-07-25 00:00:00|            12092|PENDING_PAYMENT|
|      41|2013-07-25 00:00:00|             8136|PENDING_PAYMENT|
|      43|2013-07-25 00:0

In [0]:
print("Customers Count:", customers.count())

print("Orders Count:", orders.count())

print("Products Count:", products.count())

print("Order Items Count:", order_items.count())

Customers Count: 12435
Orders Count: 68883
Products Count: 1345
Order Items Count: 172198


In [0]:
dq_summary = [

("customers_null_customer_id",
 customers.filter(col("customer_id").isNull()).count()),

("customers_duplicate_ids",
 customers.groupBy("customer_id")
 .count()
 .filter(col("count") > 1)
 .count()),

("negative_order_subtotal",
 order_items.filter(col("order_item_subtotal") < 0).count()),

("invalid_product_price",
 products.filter(col("product_price") <= 0).count())

]

dq_df = spark.createDataFrame(
    dq_summary,
    ["check_name", "failed_records"]
)

dq_df.show()

+--------------------+--------------+
|          check_name|failed_records|
+--------------------+--------------+
|customers_null_cu...|             0|
|customers_duplica...|             0|
|negative_order_su...|             0|
|invalid_product_p...|             7|
+--------------------+--------------+



In [0]:
dq_df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("retail_gold.data_quality_report")

In [0]:
%sql

SELECT *
FROM retail_gold.data_quality_report

check_name,failed_records
customers_null_customer_id,0
customers_duplicate_ids,0
negative_order_subtotal,0
invalid_product_price,7


# Data Quality Summary

The data quality validation process identified:

- No null customer IDs
- No duplicate customer records
- No negative order subtotals
- 7 products with invalid pricing (price <= 0)

These checks help ensure data reliability before downstream analytics and reporting.